In [ ]:
import numpy as np

In [ ]:
data = np.load("/home/akapociu/ift/interactiondynamics/data/uracil.npz")

for k in data.files:
    print(f"{k}: shape={data[k].shape}, dtype={data[k].dtype}")

z_key = "Z" if "Z" in data else "z" if "z" in data else None
if z_key:
    print("\nNumber of atoms:", len(data[z_key]))
    print("Atomic numbers:", data[z_key])

if "R" in data:
    print("\nNumber of frames:", data["R"].shape[0])
    print("Atoms per frame:", data["R"].shape[1])

if "E" in data:
    print("Energy shape:", data["E"].shape)

if "F" in data:
    print("Force shape:", data["F"].shape)

type: shape=(), dtype=<U1
code_version: shape=(), dtype=<U13
name: shape=(), dtype=<U18
theory: shape=(), dtype=<U7
R: shape=(27272, 87, 3), dtype=float64
z: shape=(87,), dtype=int64
F: shape=(27272, 87, 3), dtype=float64
F_min: shape=(), dtype=float64
F_max: shape=(), dtype=float64
F_mean: shape=(), dtype=float64
F_var: shape=(), dtype=float64
r_unit: shape=(), dtype=<U3
e_unit: shape=(), dtype=<U8
E: shape=(27272,), dtype=float64
E_min: shape=(), dtype=float64
E_max: shape=(), dtype=float64
E_mean: shape=(), dtype=float64
E_var: shape=(), dtype=float64
md5: shape=(), dtype=|S32
perms: shape=(32, 87), dtype=int64

Number of atoms: 87
Atomic numbers: [6 6 6 6 6 6 8 6 6 6 6 6 6 8 6 6 6 6 6 6 8 8 6 6 8 6 6 6 6 8 8 1 8 1 8 1 1
 8 1 8 1 8 1 1 8 1 8 1 8 1 8 1 1 8 1 8 1 8 1 8 1 1 8 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1]

Number of frames: 27272
Atoms per frame: 87
Energy shape: (27272,)
Force shape: (27272, 87, 3)


In [ ]:
print(data["r_unit"])
print(data["e_unit"])

Ang
kcal/mol


How are we going to turn molecules into event data... what constitutes an interaction?
First frame
Minimum distance is .92 angstroms apart... normal
- .00001 = prob broken, 5.0 = too spread out in a weird way
Max distance is 13.7 angstroms (normal yay) -> whole molecular diameter is around 14 angstroms
- 1.5 = very local/maybe bonds, 2.5 = local, 3-4 = braoder neighborhood 
First 5 force norms [ 80.01599434  29.6827551   35.26592584  67.57227943 100.41658281]
- First 5 atoms are feeling forces of those magnitiutes at frame 0
- Molecule is not at a perfectly relaxed equilibrium, there are meaningful pushes and pulls

Ok so anyways, back to interactions:

Distance Cutoff -> Prob simplest
- Atom pairs closer than cutoff are "interacting?" -> those become events for frame t?
- ^^ Decides how many edges per frame, how stable the graph is over time, whether atoms enter/leave interaction range, how noisy edge construction becomes if you perturb positions

K-nearest neighbors
- For each atom, connect it to its k-closest atoms
- Good if cutoff graphs get messy ^^
- Can be less physically intuitive, but good for stable graph size

Distance-change events
- Create an event when a pair's distance changes "enough" from one frame to the next 
- Focuses on motion, knows when something meaningful is changing, more event like?
- Can be noisy unless closeness is also used
- Only consider pairs that are already sort of close, then create events when their distance changes a lot?

Force-based edges
- Connect atoms if the force relationship between them is strong enough
- Makes sense duh but we don't have atom to atom force, only atom to everybody force. 
- Would have to do something fancy... but it's an option!

Distance Gate
- Consider atom pairs within some max radius and create events when they enter or leave the radius/ their distance changes by some threshold
- Gives physical location, temporal events, not crazy dense

In [ ]:
R = data["R"] # grab position array (all frames, all atoms, all xyz coords)
z = data["z"] # grab atom identities 
F = data["F"] # grab all forces
E = data["E"] # grab all energies

t = 0 # first timeframe aka molecule at very first instance
coords = R[t]   # (87, 3)

# compute distance from every atom to every other atom (87x87 matrix)
# dists[i, j] = distance between atom i and atom j
dists = np.linalg.norm(coords[:, None, :] - coords[None, :, :], axis=-1)

print("frame 0 coords shape:", coords.shape)
print("distance matrix shape:", dists.shape)
print("min nonzero distance:", dists[dists > 0].min()) # what is the smallest distance between two diff atoms?
print("max distance:", dists.max()) # how spread out is the whole molecule 
print("frame 0 energy:", E[t]) # total energy at frame 0
print("frame 0 first 5 force norms:", np.linalg.norm(F[t][:5], axis=1)) # computes magnitude of the force vectors for the first 5 atoms
#^^ each force is 3D (x y and z force) so norm turns this into one number -> how strong is the force on this atom

frame 0 coords shape: (87, 3)
distance matrix shape: (87, 87)
min nonzero distance: 0.9270638366369385
max distance: 13.716578486503112
frame 0 energy: -1578825.07537
frame 0 first 5 force norms: [ 80.01599434  29.6827551   35.26592584  67.57227943 100.41658281]
